# Map sequencing reads to wells

**What it does.** Read the amplicon FASTQs and count which guide RNA landed in which well.

**When to use it.** For pooled CRISPR screens, once sequencing comes back. This is what turns a plate of images into a genotype-to-phenotype table.

**What you get.** A per-well barcode count table, plus QC on read quality and consensus.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.sequencing.generate_barecode_mapping`

```
generate_barecode_mapping(settings=None)
```

Turn a folder of pooled-screen FASTQ files into per-well sgRNA count tables usable by :func:`spacr.ml.perform_regression`.

In [ ]:
from spacr.sequencing import generate_barecode_mapping

## 3. Settings

`spacr.settings.get_map_barcodes_default_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_map_barcodes_default_settings

defaults = get_map_barcodes_default_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (str) - Path to a CSV of screen/plate barcodes for the legacy
    # barcode-mapping helper. Nothing in the current code reads this
    # key: get_map_barcodes_default_settings, the only place it is
    # defined, is never called by any pipeline, so setting it has no
    # effect. The live equivalents consumed by generate_barecode_mapping
    # are row_csv, column_csv and grna_csv.
    'barcodes': '/media/carruthers/mnt3/claude/repo/spacr/resources/data/barcodes_column.csv',

    # (str) - Path to a CSV of gRNA barcode sequences for the legacy
    # barcode-mapping helper. Like 'barcodes' it exists only in
    # get_map_barcodes_default_settings, which no pipeline calls, so
    # changing it has no effect on any run. The live equivalent, read by
    # generate_barecode_mapping, is grna_csv.
    'grna': '/media/carruthers/mnt3/claude/repo/spacr/resources/data/barcodes_grna.csv',

    # (str) - Negative control identifier.
    'nc': 'TGGT1_233460_4',

    # (str) - Location of the negative control in the images.
    'nc_loc': 'c1',

    # (str) - Positive control identifier.
    'pc': 'TGGT1_220950_1',

    # (str) - Location of the positive control in the images.
    'pc_loc': 'c2',

    # (str) - Intended to map acquisition folder names to plate IDs,
    # e.g. "{'EO1': 'plate1'}", but nothing reads
    # settings['plate_dict']. Plate identity comes from the filename
    # regex or the folder name via _extract_filename_metadata instead.
    # Kept only so old settings CSVs still load.
    'plate_dict': "{'EO1': 'plate1', 'EO2': 'plate2', 'EO3': 'plate3', 'EO4': 'plate4', 'EO5': 'plate5', 'EO6': 'plate6', 'EO7': 'plate7', 'EO8': 'plate8'}",

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (bool) - In classifier training, run the held-out evaluation pass
    # (combine with train, or use alone to score an existing model). In
    # the sequencing barcode mapper it means something different:
    # process only the first read chunk and print a preview, so you can
    # sanity-check the regex and barcode CSVs in seconds. Default False.
    'test': False,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table at the start of mask generation, the
    # channel and Cellpose-model choices per object type, per-table row
    # counts and how many objects survive the nuclei/pathogen-per-cell
    # filters when measurement tables are merged, and extra
    # loader/diagnostic output in the training and UMAP paths. It only
    # adds console output, so turn it on when object counts come out
    # unexpected and you need to see which stage removed them. Defaults
    # are per-pipeline: True for mask generation, UMAP, screen analysis,
    # barcode mapping, Cellpose training and plaque analysis; False for
    # measure-and-crop, plot-from-db and plot-from-CSV, the endodyogeny
    # and class-proportion helpers, the Cellpose check/finetune tools,
    # and the screen regression, whose verbose branch display()s the
    # whole per-object score table.
    'verbose': True,

}

# Fill in anything left unset, then check the source path.
settings = get_map_barcodes_default_settings(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
generate_barecode_mapping(settings)

## Where the output went

A per-well barcode count table, plus QC on read quality and consensus.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.